# Filter long cells
Flags cells whose area exceeds `AREA_THRESHOLD × mean_area` per FOV.
Reads the flat `*_masks.tif` files produced by `omnipose_segment.ipynb` — no subfolders needed.

**Workflow:**
1. Run cell 1 to set paths and threshold.
2. Run cell 2 — detects flagged cells, saves a per-FOV long-cell mask + detail table, and a global summary.
3. Run cell 3 — area-distribution plots; move the threshold up/down in cell 1 and re-run cells 2–3 until it looks right.
4. Run cell 4 — napari inspection for any chosen FOV.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import tifffile
from skimage.measure import regionprops
from skimage.measure import perimeter as sk_perimeter

# ── USER SETTINGS ────────────────────────────────────────────────────────────
_base      = Path.home() / "Box/Zohar_Persky/projects/p2f-revisions/morph-omnipose"
IMAGE_DIR  = _base / "phase-proj"
SEG_DIR    = _base / "omnipose_seg"
OUTPUT_DIR = _base / "long_cell_filter"

AREA_THRESHOLD = 2.0   # flag cells with area > AREA_THRESHOLD × mean_area per FOV
# ─────────────────────────────────────────────────────────────────────────────

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

mask_paths = sorted(SEG_DIR.glob("*.phase_masks.tif"))
print(f"Found {len(mask_paths)} FOV(s)")
for p in mask_paths:
    print(f"  {p.name}")

In [ ]:
# ── Detect long cells, save per-FOV mask + detail table ──────────────────────
def _get_region_props(seg_mask):
    cell_props = regionprops(seg_mask)
    prop_names = ['label', 'area', 'perimeter', 'eccentricity',
                  'major_axis_length', 'minor_axis_length', 'centroid',
                  'orientation', 'solidity', 'convex_area']
    data = []
    for prop in cell_props:
        cell_data = {p: getattr(prop, p) for p in prop_names}
        convex_perim = sk_perimeter(
            np.pad(prop.convex_image, 1, mode='constant', constant_values=0)
        )
        cell_data['area_ratio']      = prop.area / prop.convex_area
        cell_data['perimeter_ratio'] = prop.perimeter / convex_perim
        data.append(cell_data)
    df = pd.DataFrame(data)
    df['perim_to_area'] = df['perimeter'] / df['area']
    df['axis_ratio']    = df['major_axis_length'] / df['minor_axis_length']
    df['circularity']   = (4 * np.pi * df['area']) / (df['perimeter'] ** 2)
    return df


all_flagged = []

for mask_path in mask_paths:
    # fov_1_hyb_1.phase_masks.tif  →  fov_1_hyb_1
    base       = Path(mask_path.stem).stem
    phase_path = IMAGE_DIR / f"{base}.phase.tif"

    if not phase_path.exists():
        print(f"  Skipping {base} — phase image not found at {phase_path}")
        continue

    mask  = tifffile.imread(mask_path).astype(np.int32)

    props_df  = _get_region_props(mask)
    mean_area = props_df['area'].mean()
    threshold = AREA_THRESHOLD * mean_area
    flagged   = props_df[props_df['area'] > threshold].copy()

    n_total   = len(props_df)
    n_flagged = len(flagged)
    print(f"{base}: {n_flagged}/{n_total} flagged  "
          f"(mean={mean_area:.1f} px, threshold={threshold:.1f} px)")

    # create per-FOV subfolder in OUTPUT_DIR
    fov_dir = OUTPUT_DIR / base
    fov_dir.mkdir(exist_ok=True)

    # mask with only the flagged cells
    long_mask = np.where(np.isin(mask, flagged['label'].values), mask, 0)
    np.save(fov_dir / f"{base}.seg.long_cells.npy", long_mask)

    # per-FOV detail table
    flagged = flagged.copy()
    flagged.insert(0, 'fov', base)
    flagged['mean_area_fov']      = round(mean_area, 1)
    flagged['threshold_used']     = round(threshold, 1)
    flagged['area_ratio_to_mean'] = (flagged['area'] / mean_area).round(2)
    cols = ['fov', 'label', 'area', 'mean_area_fov', 'threshold_used',
            'area_ratio_to_mean', 'major_axis_length', 'minor_axis_length',
            'axis_ratio', 'centroid']
    detail = flagged[cols]
    detail.to_csv(fov_dir / f"{base}.long_cells_t{AREA_THRESHOLD}.txt",
                  sep='\t', index=False)

    all_flagged.append(detail)

# global summary
summary_df   = pd.concat(all_flagged, ignore_index=True) if all_flagged else pd.DataFrame()
summary_path = OUTPUT_DIR / f"long_cells_summary_t{AREA_THRESHOLD}.txt"
summary_df.to_csv(summary_path, sep='\t', index=False)

print(f"\nTotal flagged: {len(summary_df)} cells across all FOVs")
print(f"Summary → {summary_path}")
summary_df

In [ ]:
# ── Area-distribution plots with threshold line ───────────────────────────────
import matplotlib.pyplot as plt

n     = len(mask_paths)
ncols = min(n, 3)
nrows = (n + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows), squeeze=False)

for i, mask_path in enumerate(mask_paths):
    base = Path(mask_path.stem).stem
    ax   = axes[i // ncols][i % ncols]

    mask     = tifffile.imread(mask_path).astype(np.int32)
    props_df = _get_region_props(mask)
    mean_area = props_df['area'].mean()
    threshold = AREA_THRESHOLD * mean_area
    flagged   = props_df[props_df['area'] > threshold]

    ax.hist(props_df['area'], bins=30, color='steelblue', alpha=0.7)
    ax.axvline(mean_area, color='gray', linestyle=':',
               label=f'mean ({mean_area:.0f} px)')
    ax.axvline(threshold, color='red',  linestyle='--',
               label=f'threshold ({AREA_THRESHOLD}× mean = {threshold:.0f} px)')
    if len(flagged):
        ymax = ax.get_ylim()[1]
        ax.scatter(flagged['area'], np.ones(len(flagged)) * ymax * 0.03,
                   color='red', zorder=5, s=40, label=f'{len(flagged)} flagged')
    ax.set_title(base, fontsize=9)
    ax.set_xlabel('area (px)')
    ax.set_ylabel('# cells')
    ax.legend(fontsize=7)

# hide unused subplots
for j in range(i + 1, nrows * ncols):
    axes[j // ncols][j % ncols].set_visible(False)

plt.suptitle(f'Cell area distribution  |  AREA_THRESHOLD = {AREA_THRESHOLD}', fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# ── Napari inspection ─────────────────────────────────────────────────────────
# Shows phase + all cells + flagged long cells for one FOV.
# Prints the flagged cell table so you know where to look (centroid = row, col).
import napari

FOV_TO_INSPECT = "fov_1_hyb_1"   # ← change to any FOV name

phase = tifffile.imread(IMAGE_DIR / f"{FOV_TO_INSPECT}.phase.tif")
mask  = tifffile.imread(SEG_DIR   / f"{FOV_TO_INSPECT}.phase_masks.tif").astype(np.int32)

long_mask_path = OUTPUT_DIR / FOV_TO_INSPECT / f"{FOV_TO_INSPECT}.seg.long_cells.npy"
long_mask = np.load(long_mask_path) if long_mask_path.exists() else np.zeros_like(mask)

# print flagged cell details
detail_path = OUTPUT_DIR / FOV_TO_INSPECT / f"{FOV_TO_INSPECT}.long_cells_t{AREA_THRESHOLD}.txt"
if detail_path.exists():
    print(pd.read_csv(detail_path, sep='\t').to_string(index=False))
else:
    print("No detail table found — run cell 2 first.")

viewer = napari.Viewer(title=f"{FOV_TO_INSPECT}  |  long-cell threshold = {AREA_THRESHOLD}×")
viewer.add_image(phase, name="phase", colormap="gray",
                 contrast_limits=[int(phase.min()), int(phase.max())])
viewer.add_labels(mask,      name="all_cells")
viewer.add_labels(long_mask, name=f"long_cells (>{AREA_THRESHOLD}× mean)")
napari.run()